# Chapter 27 — Dimensionality Reduction

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then the errata page.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/USER/from-absolute-zero/main/requirements.txt  # Colab only; skip locally

## Data

This chapter reuses `segments.csv`, created in Chapter 26. Run that notebook first, or just run the generator below — it is the same code.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(26)

# Four real customer types, plus a scatter of people who fit none of them.
specs = [("bargain hunters",  900, [ 18,  2.1,  46,  1.4]),
         ("weekly regulars", 1100, [ 34,  8.6,  12,  3.2]),
         ("bulk buyers",      700, [128,  1.4,   9,  6.1]),
         ("lapsed",           600, [ 22,  0.4,  71,  1.1])]
rows, truth = [], []
for name, n, (basket, freq, recency, lines) in specs:
    rows.append(np.c_[
        np.exp(rng.normal(np.log(basket), 0.30, n)),
        np.abs(rng.normal(freq, freq * 0.28, n)),
        np.abs(rng.normal(recency, recency * 0.30, n)),
        np.abs(rng.normal(lines, lines * 0.25, n))])
    truth += [name] * n
noise = 200
rows.append(np.c_[np.exp(rng.normal(np.log(45), 1.0, noise)),
                  np.abs(rng.normal(4, 3, noise)),
                  np.abs(rng.normal(40, 30, noise)),
                  np.abs(rng.normal(3, 2, noise))])
truth += ["unclassifiable"] * noise

X = np.vstack(rows)
df = pd.DataFrame(X, columns=["AvgBasket", "OrdersPerMonth",
                              "DaysSinceLast", "LinesPerOrder"]).round(2)
df["TrueType"] = truth
df = df.sample(frac=1, random_state=0).reset_index(drop=True)
df.to_csv("segments.csv", index=False)
print(f"wrote segments.csv: {len(df):,} customers, "
      f"{df.TrueType.nunique()} true groups "
      f"({(df.TrueType == 'unclassifiable').sum()} belong to none)")

## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session.

In [ ]:
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, roc_auc_score
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_digits
seg = pd.read_csv("segments.csv")
truth = seg.pop("TrueType").values
Xseg = StandardScaler().fit_transform(seg.values)
FEATS = list(seg.columns)
dig = load_digits()
Xd, yd = dig.data, dig.target

## The chapter code

### Block 1  (`c1.py`)

In [ ]:
# PCA finds directions of maximum variance. On four correlated behavioural
# measures, how much of the picture lives in how few directions?
p = PCA().fit(Xseg)
print(f"{'component':>10}{'variance':>11}{'cumulative':>13}")
for i, (v, c) in enumerate(zip(p.explained_variance_ratio_,
                               np.cumsum(p.explained_variance_ratio_)), 1):
    print(f"{i:>10}{v:>11.3f}{c:>13.3f}")

print(f"\nloadings: what each component is made of")
print(f"{'feature':<16}" + "".join(f"{'PC'+str(i+1):>9}" for i in range(4)))
for j, f in enumerate(FEATS):
    print(f"{f:<16}" + "".join(f"{p.components_[i, j]:>9.2f}"
                               for i in range(4)))

### Block 2  (`c2.py`)

In [ ]:
# 64 pixels per digit. How many directions does that really occupy?
Xs = StandardScaler().fit_transform(Xd)
p = PCA().fit(Xs)
cum = np.cumsum(p.explained_variance_ratio_)
print(f"{len(Xd):,} images, {Xd.shape[1]} pixels each")
for target in (0.50, 0.80, 0.90, 0.95, 0.99):
    k = int(np.searchsorted(cum, target) + 1)
    print(f"  {target:.0%} of the variance needs {k:>2} components "
          f"({k/Xd.shape[1]:.0%} of the columns)")

### Block 3  (`c3.py`)

In [ ]:
# Does compressing cost accuracy? Fit inside a pipeline so PCA is
# refitted in every fold, exactly as Chapter 16 requires.
cv = StratifiedKFold(5, shuffle=True, random_state=0)
full = cross_val_score(make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=5000)),
                       Xd, yd, cv=cv).mean()
print(f"{'components':>11}{'accuracy':>10}{'vs all 64':>11}")
for k in (5, 10, 20, 30, 40, 64):
    m = make_pipeline(StandardScaler(), PCA(n_components=k, random_state=0),
                      LogisticRegression(max_iter=5000))
    s = cross_val_score(m, Xd, yd, cv=cv).mean()
    print(f"{k:>11}{s:>10.4f}{s - full:>+11.4f}")
print(f"\nall 64 raw pixels, no PCA: {full:.4f}")

### Block 4  (`c4.py`)

In [ ]:
# PCA is fitted on data, so it leaks like any other transformation.
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.3,
                                      random_state=0, stratify=yd)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

# WRONG: reduce everything first, then split and cross-validate
Xall = PCA(n_components=20, random_state=0).fit_transform(
           StandardScaler().fit_transform(Xd))
wrong = cross_val_score(LogisticRegression(max_iter=5000),
                        Xall, yd, cv=cv).mean()

# RIGHT: PCA inside the pipeline, refitted in every fold
right = cross_val_score(make_pipeline(StandardScaler(),
                        PCA(n_components=20, random_state=0),
                        LogisticRegression(max_iter=5000)),
                        Xd, yd, cv=cv).mean()
print(f"PCA fitted on all data first (WRONG): {wrong:.4f}")
print(f"PCA inside the pipeline     (right): {right:.4f}")
print(f"difference: {wrong - right:+.4f}")
print("\nthe gap is inside the noise, because PCA never sees y.")
print("compare: target encoding fitted the same way (Chapter 24)")
print("inflated training AUC, and SMOTE (Chapter 23) inflated 9x.")
print("how much a transformation leaks depends on whether it")
print("uses the target.")

### Block 5  (`c5.py`)

In [ ]:
# t-SNE preserves neighbourhoods, not distances. It is a viewing tool.
import time
Xs = StandardScaler().fit_transform(Xd)
pca2 = PCA(n_components=2, random_state=0).fit_transform(Xs)

t0 = time.perf_counter()
ts2 = TSNE(n_components=2, init="pca", perplexity=30,
           random_state=0).fit_transform(Xs)
t_tsne = time.perf_counter() - t0

# How well does each 2-D view separate the ten digits? Cluster the
# view and see how far the clusters agree with the true labels.
views = [("PCA (2 components)", pca2),
         ("t-SNE (2 dimensions)", ts2)]
for name, emb in views:
    lab = KMeans(10, n_init=10, random_state=0).fit_predict(emb)
    ari = adjusted_rand_score(yd, lab)
    print(f"{name:<24} ARI against the true digit {ari:.3f}")

print(f"\nt-SNE took {t_tsne:.1f}s for {len(Xd):,} points, and cannot")
print("transform new points -- there is no .transform() to call.")

### Block 6  (`c6.py`)

In [ ]:
# Chapter 26 clustered in the original space. Does reducing first help?
cases = [("original 4 features", Xseg)]
for k in (2, 3):
    cases.append((f"PCA, {k} components",
                  PCA(n_components=k, random_state=0).fit_transform(Xseg)))
for name, data in cases:
    km = KMeans(4, n_init=10, random_state=0).fit(data)
    print(f"{name:<22} ARI {adjusted_rand_score(truth, km.labels_):.3f}")

# And on the digits, where the original space is much larger.
Xs = StandardScaler().fit_transform(Xd)
for name, data in [("original 64 pixels", Xs),
                   ("PCA, 10 components",
                    PCA(n_components=10, random_state=0).fit_transform(Xs)),
                   ("PCA, 20 components",
                    PCA(n_components=20, random_state=0).fit_transform(Xs))]:
    km = KMeans(10, n_init=10, random_state=0).fit(data)
    print(f"{name:<22} ARI {adjusted_rand_score(yd, km.labels_):.3f}")